# Etapa 3: procesamiento distribuido con PySpark

En esta sección utilizamos la API de RDDs de Apache Spark para resolver preguntas de negocio que serían ineficientes o impracticables en el motor relacional de PostgreSQL.

In [1]:
# inicialización del entorno
from pyspark import SparkContext
import re
import itertools

# como estamos en el contenedor oficial de docker, la configuración es directa
sc = SparkContext("local[*]", "AnalisisDelivery")

## 1. Análisis léxico de ingredientes (frecuencia de palabras clave)

**Pregunta de negocio:** ¿Cuáles son los ingredientes o conceptos más recurrentes en las descripciones de los platos que ofrecen los restaurantes? (Útil para tendencias de consumo).


In [7]:
# cargamos el archivo de platos usando la ruta absoluta del contenedor
rdd_platos = sc.textFile("/home/jovyan/work/platos.csv")
encabezado_platos = rdd_platos.first()

def limpiar_y_tokenizar(linea):
    campos = linea.split('|')
    # aseguramos que la línea tenga la columna de descripción (índice 3)
    if len(campos) > 3:
        descripcion = campos[3]
        # removemos puntuación y pasamos a minúsculas
        texto_limpio = re.sub(r'[^\w\s]', '', descripcion.lower())
        palabras = texto_limpio.split()
        
        # lista de palabras ignoradas (stopwords)
        stopwords = {'con', 'de', 'y', 'el', 'la', 'en', 'al', 'los', 'las', 'un', 'una', 'para'}
        # Y en tu filter te aseguras de sacar tanto las normales como estas
        return [p for p in palabras if p not in stopwords and len(p) > 2]
    return []

# construcción del pipeline
ingredientes_frecuentes = rdd_platos.filter(lambda linea: linea != encabezado_platos) \
    .flatMap(limpiar_y_tokenizar) \
    .map(lambda palabra: (palabra, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

# ejecución de la acción
print("Top 10 ingredientes/palabras en descripciones:")
for ingrediente, cantidad in ingredientes_frecuentes.take(10):
    print(f"- {ingrediente}: {cantidad} menciones")

Top 10 ingredientes/palabras en descripciones:
- elección: 24 menciones
- tomate: 23 menciones
- corte: 20 menciones
- doble: 18 menciones
- piezas: 16 menciones
- palta: 15 menciones
- masa: 12 menciones
- extra: 12 menciones
- frita: 12 menciones
- cortada: 12 menciones


## 2. Filtro colaborativo: pares de restaurantes (market basket analysis)

**Pregunta de negocio:** ¿Qué restaurantes suelen compartir la misma clientela? Esto permite generar asociaciones estratégicas o recomendar a un usuario un restaurante basándonos en lo que comen personas con gustos similares.


In [3]:
# cargamos el archivo de pedidos usando la ruta absoluta del contenedor
rdd_pedidos = sc.textFile("/home/jovyan/work/pedidos.csv")
encabezado_pedidos = rdd_pedidos.first()

# obtenemos visitas únicas por usuario
# formato de pedidos: id_pedido|id_usuario|id_restaurante|id_repartidor|id_promocion...
visitas_unicas = rdd_pedidos.filter(lambda linea: linea != encabezado_pedidos) \
    .map(lambda linea: linea.split('|')) \
    .map(lambda campos: (int(campos[1]), int(campos[2]))) \
    .distinct()

# agrupamos el historial y calculamos combinaciones
pares_restaurantes = visitas_unicas.groupByKey() \
    .mapValues(list) \
    .filter(lambda x: len(x[1]) > 1) \
    .flatMap(lambda x: itertools.combinations(sorted(x[1]), 2)) \
    .map(lambda par: (par, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

print("Top 5 pares de restaurantes consumidos por los mismos clientes:")
for par, coincidencias in pares_restaurantes.take(5):
    print(f"restaurantes {par[0]} y {par[1]} -> {coincidencias} usuarios en común")

Top 5 pares de restaurantes consumidos por los mismos clientes:
restaurantes 3 y 18 -> 22 usuarios en común
restaurantes 10 y 21 -> 21 usuarios en común
restaurantes 14 y 30 -> 20 usuarios en común
restaurantes 25 y 38 -> 20 usuarios en común
restaurantes 5 y 27 -> 19 usuarios en común


## 3. Identificación de "cazadores de promociones"

**Pregunta de negocio:** ¿Qué usuarios tienen la mayor tasa de uso de cupones de descuento respecto a su cantidad total de compras? (Condición: mínimo 3 pedidos realizados).


In [4]:
def analizar_promociones(linea):
    campos = linea.split('|')
    id_usuario = int(campos[1])
    # verificamos si la columna id_promocion (índice 4) tiene datos
    id_promocion = campos[4].strip()
    
    # si no está vacío y no dice NULL, asumimos que usó promoción
    uso_promo = 1 if (id_promocion and id_promocion.upper() != 'NULL') else 0
    
    return (id_usuario, (1, uso_promo))

tasa_promociones = rdd_pedidos.filter(lambda linea: linea != encabezado_pedidos) \
    .map(analizar_promociones) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .filter(lambda tupla: tupla[1][0] >= 3) \
    .mapValues(lambda totales: round((totales[1] / totales[0]) * 100, 2)) \
    .sortBy(lambda x: x[1], ascending=False)

print("top 5 usuarios cazadores de ofertas (porcentaje de pedidos con promo):")
for usuario, porcentaje in tasa_promociones.take(5):
    print(f"usuario {usuario}: usó cupones en el {porcentaje}% de sus pedidos")

top 5 usuarios cazadores de ofertas (porcentaje de pedidos con promo):
usuario 438: usó cupones en el 100.0% de sus pedidos
usuario 144: usó cupones en el 100.0% de sus pedidos
usuario 675: usó cupones en el 100.0% de sus pedidos
usuario 97: usó cupones en el 85.71% de sus pedidos
usuario 645: usó cupones en el 83.33% de sus pedidos
